# CAI4201 Natural Language Processing: Named Entity Recognition

This notebook compares five NER approaches on the supplied CoNLL-2003 data. The implementation uses the training split for learning, validation for model selection, and the test split only for final evaluation. Exact entity-span precision, recall, and F1 are used throughout.

## 1. Project introduction

The task is to assign a BIO NER label to every token, reconstruct entity spans, and compare the predicted spans with the gold spans. The four entity types are PER, LOC, ORG, and MISC.

## 2. Imports and configuration

In [ ]:
from pathlib import Path
from collections import Counter
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
candidates = [candidate / "data" for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "data" / "train.txt").is_file()]
if not candidates:
    candidates = [match.parent for match in Path.cwd().rglob("data/train.txt")]
if not candidates:
    raise FileNotFoundError("Could not locate data/train.txt. Open this notebook from the project folder.")
DATA_DIR = candidates[0]
ROOT = DATA_DIR.parent
RESULTS_DIR = ROOT / "results"
VISUALIZATIONS_DIR = RESULTS_DIR / "visualizations"
RESULTS_DIR.mkdir(exist_ok=True)
VISUALIZATIONS_DIR.mkdir(exist_ok=True)
RUN_EXPERIMENTS = True  # run all five experiments
ENTITY_TYPES = ["PER", "LOC", "ORG", "MISC"]

: 

## 3. Dataset loading and exploration

Each non-empty line contains a word, POS tag, chunk tag, and NER tag. Empty lines separate sentences.

In [ ]:
def read_conll(filename, keep_features=False):
    sentences = []
    current = [[], [], [], []]
    with open(DATA_DIR / filename, encoding="utf-8") as file:
        for raw_line in file:
            parts = raw_line.strip().split()
            if not parts:
                if current[0]:
                    sentences.append(tuple(current))
                    current = [[], [], [], []]
                continue
            if parts[0] == "-DOCSTART-":
                continue
            if len(parts) < 4:
                raise ValueError(f"Malformed row in {filename}: {raw_line!r}")
            for index in range(4):
                current[index].append(parts[index])
    if current[0]:
        sentences.append(tuple(current))
    return sentences if keep_features else [(tokens, ner) for tokens, _, _, ner in sentences]


train_full, valid_full, test_full = [read_conll(name, True) for name in ("train.txt", "valid.txt", "test.txt")]
train, valid, test = [[(tokens, ner) for tokens, _, _, ner in split] for split in (train_full, valid_full, test_full)]
print("sentences:", {"train": len(train), "valid": len(valid), "test": len(test)})
print("tokens:", {name: sum(len(tokens) for tokens, _ in split) for name, split in (("train", train), ("valid", valid), ("test", test))})
print("training vocabulary:", len({token for tokens, _ in train for token in tokens}))
print("maximum sentence length:", max(len(tokens) for tokens, _ in train + valid + test))

## 4. Data preprocessing

The checks below verify that all token-level fields remain aligned and that the supplied sentence splits are unchanged.

In [ ]:
def validate_alignment(split):
    for sentence_index, (tokens, labels) in enumerate(split):
        if len(tokens) != len(labels):
            raise ValueError(f"Sentence {sentence_index} has misaligned tokens and labels")


for split in (train, valid, test):
    validate_alignment(split)
for split in (train_full, valid_full, test_full):
    for tokens, pos, chunk, ner in split:
        if not len(tokens) == len(pos) == len(chunk) == len(ner):
            raise ValueError("Token, POS, chunk, and NER lengths differ")
print("alignment checks passed")

## 5. Label encoding and BIO tagging

Labels are encoded once for the sequence models. BIO validation checks that every I-TYPE continues the same entity type.

In [ ]:
tags = sorted({tag for _, labels in train for tag in labels})
tag2id = {tag: index for index, tag in enumerate(tags)}


def validate_bio(split):
    errors = []
    for sentence_index, (_, labels) in enumerate(split):
        previous_type = None
        for token_index, label in enumerate(labels):
            if label == "O":
                previous_type = None
            elif label.startswith("B-"):
                previous_type = label[2:]
            elif label.startswith("I-"):
                current_type = label[2:]
                if current_type != previous_type:
                    errors.append((sentence_index, token_index, label))
                previous_type = current_type
            else:
                errors.append((sentence_index, token_index, label))
                previous_type = None
    return errors


bio_errors = validate_bio(train)
print("labels:", tags)
print("BIO errors:", len(bio_errors))
assert not bio_errors

## 6. Shared evaluation functions

An entity is correct only when its start, end, and type all match. This is span-level evaluation, not token accuracy.

In [ ]:
def get_entities(labels):
    entities = set()
    start, entity_type = None, None
    for index, label in enumerate(list(labels) + ["O"]):
        starts_new = label.startswith("B-") or (label.startswith("I-") and label[2:] != entity_type)
        ends_current = label == "O" or not label.startswith("I-")
        if starts_new:
            if start is not None:
                entities.add((start, index - 1, entity_type))
            start, entity_type = index, label[2:]
        elif ends_current:
            if start is not None:
                entities.add((start, index - 1, entity_type))
            start, entity_type = None, None
    return entities


def evaluate(gold_sequences, predicted_sequences):
    if len(gold_sequences) != len(predicted_sequences):
        raise ValueError("Sentence counts differ")
    counts = {kind: {"tp": 0, "fp": 0, "fn": 0} for kind in ENTITY_TYPES}
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        if len(gold) != len(predicted):
            raise ValueError("Token counts differ")
        gold_entities, predicted_entities = get_entities(gold), get_entities(predicted)
        correct = gold_entities & predicted_entities
        for _, _, kind in correct:
            counts[kind]["tp"] += 1
        for _, _, kind in predicted_entities - correct:
            if kind in counts:
                counts[kind]["fp"] += 1
        for _, _, kind in gold_entities - correct:
            if kind in counts:
                counts[kind]["fn"] += 1
    total = {key: sum(item[key] for item in counts.values()) for key in ("tp", "fp", "fn")}
    rows = []
    for kind in ENTITY_TYPES + ["Overall"]:
        values = counts.get(kind, total)
        precision = values["tp"] / (values["tp"] + values["fp"]) if values["tp"] + values["fp"] else 0.0
        recall = values["tp"] / (values["tp"] + values["fn"]) if values["tp"] + values["fn"] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"entity": kind, "precision": precision, "recall": recall, "f1": f1, "support": values["tp"] + values["fn"]})
    return pd.DataFrame(rows)


## 7. Experiment 1 - N-gram model

The feature model uses a two-token context window, POS, chunk, capitalization, and a local bigram. An SGD classifier with logistic loss predicts one BIO tag per token using argmax decoding. The shared span evaluator then measures exact entity boundaries and types.

In [ ]:
def ngram_features(tokens, pos, chunk, index):
    features = {}
    for position in range(max(0, index - 2), min(len(tokens), index + 3)):
        word = tokens[position].lower()
        if position == index:
            features[f"word={word}"] = 1
            features[f"pos={pos[index]}"] = 1
            features[f"chunk={chunk[index]}"] = 1
            features["capitalized"] = int(tokens[index][:1].isupper())
            if index:
                features[f"bigram={tokens[index - 1].lower()}_{word}"] = 1
        else:
            features[f"context{position - index}={word}"] = 1
    return features


def run_ngram(train_full, prediction_split):
    x_train = [ngram_features(tokens, pos, chunk, index) for tokens, pos, chunk, _ in train_full for index in range(len(tokens))]
    y_train = [label for _, _, _, labels in train_full for label in labels]
    vectorizer = DictVectorizer()
    classifier = SGDClassifier(loss="log_loss", alpha=1e-6, average=True, max_iter=20, tol=None, random_state=SEED)
    classifier.fit(vectorizer.fit_transform(x_train), y_train)
    probabilities = []
    for tokens, pos, chunk, _ in prediction_split:
        features = [ngram_features(tokens, pos, chunk, index) for index in range(len(tokens))]
        probabilities.append(classifier.predict_proba(vectorizer.transform(features)))
    transition = np.zeros((len(classifier.classes_) + 1, len(classifier.classes_)))
    return probabilities, list(classifier.classes_), transition

## 8. Experiment 2 - Word2Vec + classifier

Word2Vec is trained on training sentences only. Each token uses the concatenated vectors from itself and its immediate neighbors. Missing vectors are zero-filled and should be reported as an OOV diagnostic.

In [ ]:
def embedding_context_features(model, tokens, index, lowercase=False, dimension=100):
    vectors = []
    for position in range(index - 1, index + 2):
        if position < 0 or position >= len(tokens):
            vectors.append(np.zeros(dimension))
            continue
        key = tokens[position].lower() if lowercase else tokens[position]
        vectors.append(model[key] if key in model else np.zeros(dimension))
    return np.concatenate(vectors)


def train_embedding_classifier(embedding_model, lowercase=False):
    x_train = [
        embedding_context_features(embedding_model, tokens, index, lowercase)
        for tokens, _ in train
        for index in range(len(tokens))
    ]
    y_train = [label for _, labels in train for label in labels]
    classifier = SGDClassifier(
        loss="log_loss", class_weight="balanced", alpha=1e-4,
        max_iter=15, tol=None, random_state=SEED
    )
    classifier.fit(np.asarray(x_train), y_train)
    return classifier


def predict_embedding_classifier(classifier, embedding_model, split, lowercase=False):
    probabilities = []
    for tokens, _ in split:
        features = [
            embedding_context_features(embedding_model, tokens, index, lowercase)
            for index in range(len(tokens))
        ]
        probabilities.append(classifier.predict_proba(np.asarray(features)))
    return probabilities


def run_word2vec_experiment():
    from gensim.models import Word2Vec
    model = Word2Vec(
        [tokens for tokens, _ in train],
        vector_size=100, window=5, min_count=2, epochs=5,
        workers=4, seed=SEED
    )
    classifier = train_embedding_classifier(model.wv)
    return model, classifier


## 9. Experiment 3 - GloVe + classifier

GloVe uses the same context-vector classifier design as Word2Vec. The pretrained `glove-wiki-gigaword-100` model is loaded through gensim and is not trained on the assignment test data.

In [ ]:
def run_glove_experiment():
    from gensim import downloader
    model = downloader.load("glove-wiki-gigaword-100")
    classifier = train_embedding_classifier(model, lowercase=True)
    return model, classifier


def report_oov_coverage(model, split, lowercase=False):
    total = sum(len(tokens) for tokens, _ in split)
    known = sum(
        1 for tokens, _ in split
        for token in tokens
        if (token.lower() if lowercase else token) in model
    )
    return {"known_tokens": known, "total_tokens": total, "oov_rate": 1 - known / total}


In [ ]:
if RUN_EXPERIMENTS:
    w2v_oov = report_oov_coverage(word2vec.wv, test, lowercase=False)
    glove_oov = report_oov_coverage(glove, test, lowercase=True)
    print(f"Word2Vec OOV rate on test: {w2v_oov['oov_rate']:.1%}")
    print(f"GloVe    OOV rate on test: {glove_oov['oov_rate']:.1%}")


## 10. Experiment 4 - LSTM with pretrained embeddings

The input is the concatenation of local Word2Vec and GloVe vectors. Dynamic right-padding preserves every original token. Sample weights exclude padded positions from the loss, and validation loss controls early stopping.

In [ ]:
def pad_feature_sequences(feature_sequences, max_len):
    dimension = feature_sequences[0].shape[1]
    values = np.zeros((len(feature_sequences), max_len, dimension), dtype=np.float32)
    mask = np.zeros((len(feature_sequences), max_len), dtype=np.float32)
    for row, sequence in enumerate(feature_sequences):
        values[row, :len(sequence)] = sequence
        mask[row, :len(sequence)] = 1.0
    return values, mask


def fit_pretrained_lstm(x_train, y_train, mask_train, x_valid, y_valid, mask_valid):
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, LSTM, Masking
    tf.random.set_seed(SEED)
    model = Sequential([
        Masking(mask_value=0.0, input_shape=x_train.shape[1:]),
        LSTM(64, return_sequences=True),
        Dense(len(tags), activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(
        x_train, y_train, sample_weight=mask_train,
        validation_data=(x_valid, y_valid, mask_valid),
        epochs=20, batch_size=512,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=1,
    )
    return model, history


def predict_unpadded(model, values, split):
    probabilities = model.predict(values, verbose=0)
    return [probabilities[row, :len(tokens)] for row, (tokens, _) in enumerate(split)]


## 11. Experiment 5 - LSTM with trainable embedding layer

The trainable embedding model maps the training vocabulary to IDs, uses `mask_zero=True`, and predicts every original test token after removing only batch padding.

In [ ]:
def build_word_ids(split, vocabulary, max_len):
    values = np.zeros((len(split), max_len), dtype=np.int32)
    mask = np.zeros((len(split), max_len), dtype=np.float32)
    for row, (tokens, _) in enumerate(split):
        ids = [vocabulary.get(token, 1) for token in tokens]
        values[row, :len(ids)] = ids
        mask[row, :len(ids)] = 1.0
    return values, mask


def build_label_ids(split, max_len):
    values = np.zeros((len(split), max_len), dtype=np.int32)
    for row, (_, labels) in enumerate(split):
        values[row, :len(labels)] = [tag2id[label] for label in labels]
    return values


def fit_trainable_lstm(x_train, y_train, mask_train, x_valid, y_valid, mask_valid, vocabulary_size):
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Embedding, LSTM
    tf.random.set_seed(SEED)
    model = Sequential([
        Embedding(vocabulary_size, 100, mask_zero=True),
        LSTM(64, return_sequences=True),
        Dense(len(tags), activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(
        x_train, y_train, sample_weight=mask_train,
        validation_data=(x_valid, y_valid, mask_valid),
        epochs=20, batch_size=512,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=1,
    )
    return model, history


## 12. Final comparison of all five experiments

When `RUN_EXPERIMENTS` is enabled, each model must produce validation and test predictions, which are passed through the shared `evaluate` function. Test results are not used for model selection.

In [ ]:
def wide_result_row(model, split, report):
    row = {"model": model, "split": split}
    for _, values in report.iterrows():
        prefix = "overall" if values.entity == "Overall" else values.entity.lower()
        row[f"{prefix}_precision"] = values.precision
        row[f"{prefix}_recall"] = values.recall
        row[f"{prefix}_f1"] = values.f1
    return row


# End-to-end execution cell
# This cell is intentionally visible and is the only cell that writes final results.


def train_predict_ngram(train_full_data, prediction_full_data):
    features = [ngram_features(tokens, pos, chunk, i) for tokens, pos, chunk, _ in train_full_data for i in range(len(tokens))]
    labels = [label for _, _, _, sequence in train_full_data for label in sequence]
    vectorizer = DictVectorizer()
    classifier = SGDClassifier(loss="log_loss", alpha=1e-6, average=True, max_iter=20, tol=None, random_state=SEED)
    classifier.fit(vectorizer.fit_transform(features), labels)
    predictions = []
    for tokens, pos, chunk, _ in prediction_full_data:
        current = [ngram_features(tokens, pos, chunk, i) for i in range(len(tokens))]
        predictions.append(classifier.predict(vectorizer.transform(current)).tolist())
    return predictions


def train_predict_embedding(model, lowercase, training_data, prediction_data):
    classifier = train_embedding_classifier(model, lowercase)
    probabilities = predict_embedding_classifier(classifier, model, prediction_data, lowercase)
    return [[classifier.classes_[index] for index in np.argmax(values, axis=1)] for values in probabilities]


def make_lstm_labels(split, max_len):
    labels = np.zeros((len(split), max_len), dtype=np.int32)
    mask = np.zeros((len(split), max_len), dtype=np.float32)
    for row, (_, sequence) in enumerate(split):
        labels[row, :len(sequence)] = [tag2id[label] for label in sequence]
        mask[row, :len(sequence)] = 1.0
    return labels, mask


def run_lstm_models(word2vec, glove):
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Embedding, LSTM, Masking

    tf.random.set_seed(SEED)
    max_len = max(len(tokens) for split in (train, valid, test) for tokens, _ in split)
    y_train, mask_train = make_lstm_labels(train, max_len)
    y_valid, mask_valid = make_lstm_labels(valid, max_len)
    counts = Counter(token for tokens, _ in train for token in tokens)
    vocabulary = {"<PAD>": 0, "<UNK>": 1}
    vocabulary.update({word: index + 2 for index, (word, count) in enumerate(item for item in counts.items() if item[1] >= 2)})

    def word_ids(split):
        values = np.zeros((len(split), max_len), dtype=np.int32)
        for row, (tokens, _) in enumerate(split):
            ids = [vocabulary.get(token, 1) for token in tokens]
            values[row, :len(ids)] = ids
        return values

    def fixed_features(split):
        values = np.zeros((len(split), max_len, 600), dtype=np.float32)
        for row, (tokens, _) in enumerate(split):
            for index in range(len(tokens)):
                values[row, index] = np.concatenate([
                    embedding_context_features(word2vec.wv, tokens, index),
                    embedding_context_features(glove, tokens, index, lowercase=True),
                ])
        return values

    def fit(x_train, x_valid, trainable):
        layers = [Embedding(len(vocabulary), 100, mask_zero=True)] if trainable else [Masking(mask_value=0.0, input_shape=x_train.shape[1:])]
        layers += [LSTM(64, return_sequences=True), Dense(len(tags), activation="softmax")]
        model = Sequential(layers)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        model.fit(
            x_train, y_train, sample_weight=mask_train,
            validation_data=(x_valid, y_valid, mask_valid),
            epochs=20, batch_size=512,
            callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
            verbose=1,
        )
        return model

    def decode(model, values, split):
        output = model.predict(values, verbose=0)
        return [[tags[index] for index in np.argmax(output[row, :len(tokens)], axis=1)] for row, (tokens, _) in enumerate(split)]

    train_ids, valid_ids, test_ids = word_ids(train), word_ids(valid), word_ids(test)
    trainable = fit(train_ids, valid_ids, True)
    trainable_predictions = {
        "valid": decode(trainable, valid_ids, valid),
        "test": decode(trainable, test_ids, test),
    }

    train_features, valid_features, test_features = fixed_features(train), fixed_features(valid), fixed_features(test)
    pretrained = fit(train_features, valid_features, False)
    pretrained_predictions = {
        "valid": decode(pretrained, valid_features, valid),
        "test": decode(pretrained, test_features, test),
    }
    return {
        "LSTM + trainable embedding": trainable_predictions,
        "LSTM + pretrained embeddings": pretrained_predictions,
    }


if RUN_EXPERIMENTS:
    from gensim import downloader
    from gensim.models import Word2Vec

    word2vec, glove = None, None
    ngram_valid = train_predict_ngram(train_full, valid_full)
    ngram_test = train_predict_ngram(train_full, test_full)
    word2vec = Word2Vec([tokens for tokens, _ in train], vector_size=100, window=5, min_count=2, epochs=5, workers=4, seed=SEED)
    word2vec_valid = train_predict_embedding(word2vec.wv, False, train, valid)
    word2vec_test = train_predict_embedding(word2vec.wv, False, train, test)
    glove = downloader.load("glove-wiki-gigaword-100")
    glove_valid = train_predict_embedding(glove, True, train, valid)
    glove_test = train_predict_embedding(glove, True, train, test)
    lstm_outputs = run_lstm_models(word2vec, glove)

    prediction_sets = {
        "N-gram": {"valid": ngram_valid, "test": ngram_test},
        "Word2Vec": {"valid": word2vec_valid, "test": word2vec_test},
        "GloVe": {"valid": glove_valid, "test": glove_test},
        **lstm_outputs,
    }
    rows = []
    gold = {"valid": [labels for _, labels in valid], "test": [labels for _, labels in test]}
    for model_name, splits in prediction_sets.items():
        for split_name, predictions in splits.items():
            rows.append(wide_result_row(model_name, split_name, evaluate(gold[split_name], predictions)))
    final_results = pd.DataFrame(rows)
    final_results.to_csv(RESULTS_DIR / "results.csv", index=False)
    final_results.to_csv(RESULTS_DIR / "per_class_results.csv", index=False)
    save_visualizations(final_results)
    display(final_results[final_results.split == "test"].sort_values("overall_f1", ascending=False))
else:
    print("Set RUN_EXPERIMENTS = True to run all five experiments.")


## 13. Visualizations

The plots below are generated from the actual `results.csv` values. They should be regenerated after all five experiments have completed.

In [ ]:
def save_visualizations(results):
    test_results = results[results.split == "test"].sort_values("overall_f1", ascending=False)
    test_results.set_index("model")[["overall_precision", "overall_recall", "overall_f1"]].plot(kind="bar", ylim=(0, 1), figsize=(10, 5))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "model_performance.png", dpi=150)
    plt.close()
    test_results.set_index("model")[[f"{kind.lower()}_f1" for kind in ENTITY_TYPES]].plot(kind="bar", ylim=(0, 1), figsize=(10, 5))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "per_class_f1.png", dpi=150)
    plt.close()
    test_results.plot(x="model", y="overall_f1", kind="barh", legend=False, figsize=(8, 4))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "overall_f1_ranking.png", dpi=150)
    plt.close()

## 14. Final observations

The report should discuss the measured validation and test results, the effect of lexical context and embeddings, OOV coverage, and the difference between fixed pretrained features and a trainable embedding layer. The report and video must be written in the student's own words; this notebook supplies implementation evidence, not an AI-generated report.